[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Indexing and Slicing


## What you will be able to do

Read single readings, whole stations, single hours and ranges out of the readings with `a[i, j]` and
slices, step through them every 24 hours, and know when the piece you took still shares memory with
the array it came from.

## The idea

### The problem

Most questions are about part of the readings. What did Oslo read at noon on 1 July? What did every
station read at that same hour? What were the first week's readings, or the last day's, or every
reading taken at midnight? Each one is a single value, a row, a column or a range picked out of the
grid.

With a list of lists, a single value is `readings_list[1][4356]`, and a row is one of the inner
lists. A column, one hour at every station, has no notation of its own, and needs a loop that picks
the same position out of each list. Every 24th reading needs a loop or a slice with a step, done
once for each station.

### What indexing and slicing are

> **Indexing** picks values out of an array by position, one position for each dimension, separated by
> commas inside one pair of brackets. A **slice**, written `start:stop:step`, picks a range of
> positions along one dimension. A slice of an array is a **view**, a new array that reads the same
> memory as the original, so a change made through either one shows in both.

### Why it works that way

`readings[1, 4356]` reads row 1, column 4,356, Oslo at noon on 1 July. A colon on its own means the
whole of that dimension, so `readings[:, 4356]` is the column, every station at that hour, and
`readings[1]` with the second position left out is the whole row. Negative positions count from the
end, as they do in lists, and a slice leaves out its stop, so `0:24` is the first 24 hours and
`-24:` the last 24.

The surprise is the view. Slicing a Python list copies the values into a new list, and changing the
copy leaves the original alone. Slicing an array copies nothing. It makes a new array object that
reads part of the same memory, which is why slicing a large array costs no time at all, and why a
change written through the slice lands in the original. When you want a piece of your own to
change, `.copy()` makes one.

The same rule makes writing into a range easy. `filled[2, 1440:1464] = ...` replaces Svalbard's
silent day in place, with no loop, which is the next step after finding the range.

### Where this shows up

Cropping an image is a slice, `image[100:200, 50:150]`, and the crop shares the image's memory. The
`iloc` indexer in the **Pandas, Deep Dive** guide picks rows and columns by position in the same
way. The **Boolean Masks** and **Fancy Indexing** notebooks pick values by condition and by a list of
positions, and those two return copies.

### What this notebook covers

- one reading with `readings[i, j]`, and counting from the end
- a whole station and a whole hour, with `:` for a whole dimension
- ranges with slices, and every 24th hour with a step
- a slice as a view, against a list's slice as a copy
- `.copy()`, and `np.shares_memory` to check
- writing into a slice, to fill Svalbard's silent day

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

grid = np.arange(12).reshape(3, 4)
print(grid[1, 2])
print(grid[:, 0])
print(grid[0, 1:3])
```

```
6
[0 4 8]
[1 2]
```

Row 1, column 2 holds 6. `grid[:, 0]` is every row's first value, a column read as one array. Row 0
from column 1 up to column 3, which is left out, is 1 and 2.

## Setup

Three imports, the year of readings, and a helper that finds an hour's position.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings, and `hour_of_year` finds positions

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained, and `readings` is the same
values as one array of shape (4, 8760). `hour_of_year(7, 1, 12)` gives the position of noon on
1 July along a station's row, the number of hours since the year began.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### A single reading

In [2]:
noon_july = hour_of_year(7, 1, 12)

print(noon_july)
print(readings[1, noon_july])
print(readings[1][noon_july])
print(readings[-1, -1])

4356
17.5
17.5
-6.5


Noon on 1 July is hour 4,356 of the year, and Oslo read 17.5 degrees then. `readings[1][noon_july]`
gives the same value in two steps, the row and then the value in it, and the single pair of brackets
with a comma does it in one. `readings[-1, -1]` is the last station's last hour, Tromso at 23:00 on
31 December.

### A whole station, and a whole hour

A colon on its own takes the whole of a dimension. Leaving the last position out does the same:

In [3]:
tromso = readings[3]
at_noon = readings[:, noon_july]

print(tromso.shape)
print(at_noon.shape)
for name, value in zip(STATIONS, at_noon):
    print(f"{name:8} {value}")

(8760,)
(4,)
Bergen   19.1
Oslo     17.5
Svalbard 6.4
Tromso   14.3


`readings[:, noon_july]` is the column a list of lists has no notation for, the four stations at
noon on 1 July, as one array of four.

### Ranges with slices

A slice leaves out its stop, so `hour_of_year(7, 1)` to `hour_of_year(7, 2)` is the whole of 1 July:

In [4]:
july_first = readings[:, hour_of_year(7, 1):hour_of_year(7, 2)]

print(july_first.shape)
print(readings[0, :24].shape, readings[0, :168].shape, readings[0, -24:].shape)
print(july_first[0].round(1))

(4, 24)
(24,) (168,) (24,)
[14.6 14.5 14.5 13.  13.4 14.  14.8 15.7 15.  16.1 17.2 18.2 19.1 19.9
 18.8 19.2 19.4 19.4 19.2 18.9 16.8 16.3 15.8 15.4]


The first slice keeps all four stations and 24 hours. `:24` is the first day, `:168` the first week,
and `-24:` the last day of the year, each 24 or 168 hours long.

### Every 24th hour with a step

The third part of a slice is the step. From 0 in steps of 24 gives every midnight, and from 12 every
noon:

In [5]:
midnights = readings[0, ::24]
noons = readings[0, 12::24]

print(midnights.shape, noons.shape)
print(round(midnights.mean(), 2), round(noons.mean(), 2))

(365,) (365,)
5.88 10.12


365 of each. Bergen's noon readings average 10.12 degrees and its midnight readings 5.88, the daily
swing that `year_of_readings` puts into every day.

### A slice is a view

`np.shares_memory` tells whether two arrays read the same memory. A slice does:

In [6]:
print(np.shares_memory(july_first, readings))

True


To change values without touching `readings`, the cells below work on `scratch`, a copy of it.
Changing the first value of a slice of `scratch` changes `scratch` itself:

In [7]:
scratch = readings.copy()

first_three = scratch[0, :3]
first_three[0] = 99

print(scratch[0, :3])
print(readings[0, :3])

[99.  -3.7 -3.7]
[-3.6 -3.7 -3.7]


A list behaves differently. Its slice is a copy, and changing the copy leaves the list alone:

In [8]:
first_three_list = readings_list[0][:3]
first_three_list[0] = 99

print(readings_list[0][:3])

[-3.6, -3.7, -3.7]


### .copy() for a piece of your own

`.copy()` on a slice makes an array with memory of its own, which can change without affecting
anything else:

In [9]:
mine = readings[0, :3].copy()
mine[0] = 99

print(np.shares_memory(mine, readings))
print(mine, readings[0, :3])

False
[99.  -3.7 -3.7] [-3.6 -3.7 -3.7]


### Writing into a slice

Assigning to a slice writes into the array it came from, which is how Svalbard's silent day can be
filled in. Here a copy, `filled`, takes the readings of 1 March in place of the missing 2 March:

In [10]:
silent_start, silent_end = hour_of_year(3, 2), hour_of_year(3, 3)
filled = readings.copy()

filled[2, silent_start:silent_end] = filled[2, silent_start - 24:silent_start]

print(np.isnan(readings).sum(), np.isnan(filled).sum())
print(filled[2, silent_start:silent_start + 3])

24 0
[-12.8 -13.  -14.7]


The 24 `nan` values are gone from `filled` and still in `readings`. Repeating the day before is a
crude estimate, and it shows the mechanism, one slice on each side of `=`, of the same length, with
no loop.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/05-indexing-and-slicing-solutions.ipynb).

**1.** Print Bergen's reading at 15:00 on 9 July, using `hour_of_year`.

In [11]:
# your code here

**2.** Print every station's reading at midnight on 1 January, as one array of four.

In [12]:
# your code here

**3.** Take Oslo's readings for the whole of March with one slice, and print its shape and its mean rounded to two places.

In [13]:
# your code here

**4.** Print the mean of Tromso's readings at 06:00 every day of the year, using a step, rounded to two places.

In [14]:
# your code here

**5.** Copy the first day of Svalbard's readings, change the copy's first value to 0, and show that `readings` did not change.

In [15]:
# your code here

## Common errors

### IndexError: index 4 is out of bounds for axis 0 with size 4

In [16]:
readings[4, 0]                                             # a fifth station

IndexError: index 4 is out of bounds for axis 0 with size 4

There are four stations, at positions 0 to 3. Axis 0 is the first dimension, the stations, and its
size is 4, so 4 is one past the end. `len(STATIONS)` or `readings.shape[0]` gives the count.

### IndexError: index 8760 is out of bounds for axis 1 with size 8760

In [17]:
readings[0, HOURS]                                         # meant to be the last hour

IndexError: index 8760 is out of bounds for axis 1 with size 8760

The hours run from 0 to 8,759, so `HOURS` is one past the last. Axis 1 is the hours, and the message
gives its size. The last hour is `readings[0, HOURS - 1]`, or `readings[0, -1]`.

### IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [18]:
oslo = readings[1]

oslo[1, noon_july]

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

`oslo` is already one station's row, with a single dimension, so it takes one position. The station
was chosen when the row was taken. Write `oslo[noon_july]`, or `readings[1, noon_july]` from the
whole array.

### No error, and the original array changed through its slice

In [19]:
scratch = readings.copy()

day_in_fahrenheit = scratch[0, :24]
day_in_fahrenheit *= 9 / 5
day_in_fahrenheit += 32

print(scratch[0, :3])

[25.52 25.34 25.34]


`*=` and `+=` change an array in place, and `day_in_fahrenheit` is a view, so Bergen's first day in
`scratch` is now in Fahrenheit, mixed in among a year of Celsius. Make a new array with the
arithmetic, `scratch[0, :24] * 9 / 5 + 32`, or take a `.copy()` before changing anything in place.

### No error, and a last day one hour short

In [20]:
last_day = readings[0, -24:-1]

print(last_day.shape)
print(readings[0, -24:].shape)

(23,)
(24,)


A slice leaves out its stop, so `-24:-1` stops before the last hour and returns 23 hours. To run to
the end, leave the stop out, `-24:`.

## Recap

- Read one value with `readings[i, j]`, and count from the end with negative positions
- Take a whole station with `readings[i]` and a whole hour with `readings[:, j]`
- Take a range with `start:stop`, which leaves out its stop, and every 24th hour with a step
- Treat a slice as a view of the original, and take `.copy()` before changing a piece of it
- Write into a slice to change a range in place

## What is next

The **Boolean Masks** notebook picks readings by what they are instead of where they are, such as
every hour below freezing, without a loop.

---

&#8592; **Previous:** [Shape](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/04-shape.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
